# 06 — Historical Regime Analysis & Data Quality

This notebook has two sections:

**Part A — Data Quality**  
Per-ticker diagnostics: missing values, consecutive gaps, zero-return days,
5σ outliers, skewness/kurtosis, annualised volatility, pairwise coverage overlap.

**Part B — Historical Regime Analysis**  
How do crypto ↔ traditional-asset rolling correlations behave *across* major
market regimes? Regimes covered:
- Crypto Bull 2017 / Bear 2018
- Calm 2019
- COVID Crash & Recovery (2020)
- Crypto Bull 2021
- Rate Hike Cycle / Terra-Luna Collapse / FTX Collapse (2022)
- Crypto Recovery 2023
- BTC ETF / Halving 2024

In [ ]:
import sys, os
ROOT = r'D:\clode9'
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from thesis_app.pipeline import load_config, build_paths, fetch_prices, compute_returns
from thesis_app.notebook_helpers import apply_thesis_plot_style
from thesis_app.data_quality import run_data_quality_report, check_coverage_overlap
from thesis_app.regime_analysis import (
    REGIMES, run_regime_analysis,
    plot_rolling_corr_with_regimes, plot_regime_corr_boxplot,
    plot_all_pairs_regime_heatmap, compute_regime_stats,
)

apply_thesis_plot_style()
%matplotlib inline

cfg   = load_config(os.path.join(ROOT, 'config.yaml'))
paths = build_paths(cfg['base_dir'])

tickers = cfg['assets']['crypto'] + cfg['assets']['traditional']
prices  = fetch_prices(paths, tickers, cfg['start_date'], cfg.get('end_date'))
returns = compute_returns(paths, prices)

base   = cfg['base_asset']           # 'BTC-USD'
others = [t for t in tickers if t != base]
windows = cfg['rolling_windows']     # [14, 30, 60, 90]

print(f'Prices: {prices.shape}  {prices.index[0].date()} → {prices.index[-1].date()}')
print('Tickers:', list(prices.columns))

---
## Part A — Data Quality

In [ ]:
# ── Per-ticker quality statistics ──────────────────────────────────────────
quality_dir = os.path.join(paths.tables, 'data_quality')
dq = run_data_quality_report(prices, output_dir=quality_dir, sigma_threshold=5.0)

stats_df = dq['stats']
display(stats_df)

In [ ]:
# ── Coverage overlap matrix ─────────────────────────────────────────────────
overlap = dq['overlap']
print('Pairwise observation overlap (trading days both assets have data):')
display(overlap)

In [ ]:
# ── Missing value heatmap ───────────────────────────────────────────────────
from thesis_app.data_quality import plot_missing_heatmap, plot_return_outliers, plot_price_coverage

plot_missing_heatmap(prices)
plt.show()

In [ ]:
# ── Price coverage bars ─────────────────────────────────────────────────────
plot_price_coverage(prices)
plt.show()

In [ ]:
# ── Return outliers (5σ highlighted in red) ─────────────────────────────────
plot_return_outliers(returns, sigma_threshold=5.0)
plt.show()

# Print the actual outlier dates for BTC
ret_btc = returns[base].dropna()
z_btc   = (ret_btc - ret_btc.mean()) / ret_btc.std()
btc_outliers = ret_btc[z_btc.abs() > 5]
print(f'\nBTC-USD observations |z| > 5σ  ({len(btc_outliers)} total):')
print(btc_outliers.sort_values().to_string())

In [ ]:
# ── LaTeX table ─────────────────────────────────────────────────────────────
print(dq['latex'])

---
## Part B — Historical Regime Analysis

In [ ]:
# ── Regime catalogue overview ───────────────────────────────────────────────
import pandas as pd
regime_df = pd.DataFrame(
    [(k.replace('\n', ' '), v[0], v[1], v[3]) for k, v in REGIMES.items()],
    columns=['Regime', 'Start', 'End', 'Description']
)
display(regime_df)

In [ ]:
# ── Run full regime analysis (saves figures + tables to outputs/tables/regimes) ──
regime_dir = os.path.join(paths.tables, 'regimes')
regime_results = run_regime_analysis(
    prices, base=base, others=others,
    windows=windows, output_dir=regime_dir
)
print(f'Regime analysis done. Artefacts in: {regime_dir}')

In [ ]:
# ── Annotated rolling-correlation plots: BTC vs each asset (w=30) ───────────
w = 30
for other in others:
    if other not in returns.columns:
        continue
    corr = returns[base].rolling(w).corr(returns[other]).dropna()
    fig = plot_rolling_corr_with_regimes(
        corr, pair_label=f'{base} / {other}', window=w
    )
    plt.show()

In [ ]:
# ── Box-plots: correlation distribution per regime ──────────────────────────
for other in ['^GSPC', 'GLD']:  # key pairs for thesis discussion
    if other not in returns.columns:
        continue
    fig = plot_regime_corr_boxplot(returns, base, other, window=30)
    plt.show()

In [ ]:
# ── All-pairs × all-regimes heatmap ─────────────────────────────────────────
all_pairs = [(base, o) for o in others if o in returns.columns]
fig = plot_all_pairs_regime_heatmap(returns, all_pairs)
plt.show()

In [ ]:
# ── Per-regime statistics table for primary pair (BTC vs S&P 500) ───────────
other_primary = '^GSPC'
w = 30
if other_primary in returns.columns:
    key = (other_primary, w)
    stats_gspc = regime_results[key]['stats']
    print(f'Regime statistics: {base} vs {other_primary}, w={w}')
    display(stats_gspc)
    print('\n--- LaTeX ---')
    print(regime_results[key]['latex'])

In [ ]:
# ── Regime correlation table: BTC vs Gold ───────────────────────────────────
other_gld = 'GLD'
w = 30
if other_gld in returns.columns:
    key = (other_gld, w)
    stats_gld = regime_results[key]['stats']
    print(f'Regime statistics: {base} vs {other_gld}, w={w}')
    display(stats_gld)

In [ ]:
# ── Window sensitivity: how does regime picture change with window length? ───
other_check = '^GSPC'
if other_check in returns.columns:
    fig, axes = plt.subplots(2, 2, figsize=(18, 9), sharex=True, sharey=True)
    axes_flat = axes.flatten()
    for idx, w in enumerate(windows):
        corr = returns[base].rolling(w).corr(returns[other_check]).dropna()
        ax = axes_flat[idx]
        ax.plot(corr.index, corr.values, color='steelblue', lw=0.9)
        ax.axhline(0, color='black', lw=0.5, ls='--')
        # Shade COVID crash
        ax.axvspan(pd.Timestamp('2020-02-20'), pd.Timestamp('2020-03-23'),
                   color='#8c564b', alpha=0.25, label='COVID Crash')
        # Shade FTX collapse
        ax.axvspan(pd.Timestamp('2022-11-01'), pd.Timestamp('2022-11-30'),
                   color='#e377c2', alpha=0.25, label='FTX Collapse')
        ax.set_title(f'w = {w}d', fontsize=11, fontweight='bold')
        ax.set_ylim(-1.05, 1.05)
        ax.legend(fontsize=8)
    fig.suptitle(f'Rolling correlation {base} / {other_check} — window sensitivity',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(paths.figures, f'regime_window_sensitivity_{base.replace("-","")}_{other_check.replace("^","")}.png'),
                dpi=140)
    plt.show()

In [ ]:
# ── Key finding summary ──────────────────────────────────────────────────────
print('=== KEY FINDINGS ===')
for other in ['^GSPC', 'GLD', 'ETH-USD']:
    if other not in returns.columns:
        continue
    corr30 = returns[base].rolling(30).corr(returns[other]).dropna()
    stats = compute_regime_stats(corr30, returns[base])
    if stats.empty:
        continue
    max_row = stats.loc[stats['mean_corr'].abs().idxmax()]
    min_row = stats.loc[stats['mean_corr'].idxmin()]
    max_pos = stats.loc[stats['mean_corr'].idxmax()]
    print(f'\n{base} vs {other} (30d rolling):')
    print(f'  Highest absolute corr: {max_row["regime"]} (mean ρ = {max_row["mean_corr"]:.3f})')
    print(f'  Most positive corr:    {max_pos["regime"]} (mean ρ = {max_pos["mean_corr"]:.3f})')
    print(f'  Most negative corr:    {min_row["regime"]} (mean ρ = {min_row["mean_corr"]:.3f})')
    print(f'  Full-sample mean:      {corr30.mean():.3f}')

In [ ]:
print('✅ Regime analysis complete.')
print(f'   Figures → {paths.figures}')
print(f'   Tables  → {regime_dir}')